# 25 бинарных классификаторов: породы собак и класс «другие»

Гипотеза: для каждой из 25 пород обучается своя модель, которая отвечает только на вопрос «это моя порода или нет». Если после всех 25 инференсов ни одна модель не уверена в своей породе, кадр относится к классу «другие».

Один многоклассовый классификатор с порогом уверенности уже не смог отделить «другие». Здесь проверяется другой механизм: набор независимых бинарных моделей.

Ячейки запускаются сверху вниз. Обучение долгое; уже сохранённые чекпоинты при повторном запуске пропускаются.

Четыре тестовые группы считаются отдельно:

1. известные породы — отложенные кадры тех же 25 пород;
2. другие породы собак, которых не было при обучении;
3. похожие животные — волк и лиса;
4. не собаки.

В CSV пишутся вероятности всех 25 моделей, а не только итоговый класс.


## Зависимости

```bash
pip install torch torchvision pillow pandas matplotlib scikit-learn tqdm
```


In [ ]:
from __future__ import annotations

import json
import random
import tarfile
import urllib.request
from pathlib import Path
from urllib.parse import quote

try:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    from PIL import Image
    from sklearn.metrics import roc_auc_score
    from torch.utils.data import DataLoader, Dataset
    from torchvision import transforms as T
    from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
    from tqdm.auto import tqdm
except ImportError as exc:
    raise ImportError(
        "Не хватает зависимости. Установите: "
        "pip install torch torchvision pillow pandas matplotlib scikit-learn tqdm"
    ) from exc


def project_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "dog_breeds":
        return cwd
    return cwd / "dog_breeds"


BASE = project_dir()
DATA = BASE / "data"
STANFORD = DATA / "stanford"
IMAGES = STANFORD / "Images"
EXTERNAL = DATA / "external"
MODELS = BASE / "models" / "binary"
RESULTS = BASE / "results"
for folder in (DATA, STANFORD, EXTERNAL, MODELS, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

# Ручки эксперимента. После смены эпох, learning rate или списка пород
# поставьте FORCE_RETRAIN = True либо удалите dog_breeds/models/binary.
SEED = 42
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
NEG_PER_POS = 1.0
EPOCHS = 8
LR = 1e-4
BATCH_SIZE = 32
IMG_SIZE = 224
PATIENCE = 2
WEIGHT_DECAY = 1e-4
THRESHOLDS = (0.5, 0.7, 0.9)
TARGET_RECALL = 0.8
N_OTHER_PER_BREED = 10
INFER_BATCH = 64
FORCE_RETRAIN = False
OTHER = "другие"

# Папки Stanford Dogs. Негативы и обучение берутся только из этих 25 пород.
KNOWN_BREEDS = [
    "n02085620-Chihuahua",
    "n02085936-Maltese_dog",
    "n02086240-Shih-Tzu",
    "n02088364-beagle",
    "n02088238-basset",
    "n02088094-Afghan_hound",
    "n02088466-bloodhound",
    "n02091134-whippet",
    "n02094433-Yorkshire_terrier",
    "n02099601-golden_retriever",
    "n02099712-Labrador_retriever",
    "n02100877-Irish_setter",
    "n02102318-cocker_spaniel",
    "n02106030-collie",
    "n02106662-German_shepherd",
    "n02107142-Doberman",
    "n02108089-boxer",
    "n02108915-French_bulldog",
    "n02109047-Great_Dane",
    "n02109525-Saint_Bernard",
    "n02110185-Siberian_husky",
    "n02110958-pug",
    "n02111889-Samoyed",
    "n02112018-Pomeranian",
    "n02113799-standard_poodle",
]

USER_AGENT = "aisec-hse-dog-breeds-experiment/1.0 (student research notebook)"
IMAGES_URL = "http://vision.stanford.edu/aditya86/ImageNetDogs/images.tar"
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def short_name(folder: str) -> str:
    return folder.split("-", 1)[1]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
    print(
        "ВНИМАНИЕ: GPU не найден, обучение идёт на CPU. "
        "25 моделей MobileNetV2 могут считаться много часов. "
        "Уже сохранённые чекпоинты при повторном запуске пропускаются."
    )

print(f"device: {DEVICE}")
print(f"каталог эксперимента: {BASE}")


## Данные Stanford Dogs

Архив изображений (~757 МБ) скачивается с [официальной страницы](http://vision.stanford.edu/aditya86/ImageNetDogs/), если его ещё нет локально: `dog_breeds/data/stanford/images.tar`.

В обучение входят только 25 пород. Остальные 95 пород не используются ни как позитив, ни как негатив: это тестовая группа «другие собаки». Среди этих 95 есть динго, дхоль и гиеновидная собака — они ближе к диким псовым, чем к комнатным породам, но в этой группе остаются. Волк и лиса в Stanford Dogs не входят и лежат отдельно.

Сплит каждой из 25 пород — 70/15/15, seed 42, до сборки бинарных выборок. Тестовый кадр известной породы не попадает в обучение ни одной из 25 моделей.

Для классификатора породы *i* позитив — train этой породы, негатив — столько же кадров из train остальных 24 пород, поровну с каждой.


In [ ]:
def stanford_ready() -> bool:
    if not IMAGES.is_dir():
        return False
    folders = [path for path in IMAGES.iterdir() if path.is_dir()]
    return len(folders) >= 100


def download_file(url: str, dest: Path, min_bytes: int) -> None:
    if dest.exists() and dest.stat().st_size >= min_bytes:
        print(f"уже есть: {dest} ({dest.stat().st_size / 1e6:.1f} МБ)")
        return
    tmp = dest.with_suffix(dest.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    try:
        response_ctx = urllib.request.urlopen(request, timeout=120)
    except Exception as exc:
        raise RuntimeError(
            f"Не удалось скачать {url}. Положите архив вручную в {dest} и запустите ячейку ещё раз."
        ) from exc
    with response_ctx as response, tmp.open("wb") as handle, tqdm(
        total=int(response.headers.get("Content-Length") or 0),
        unit="B",
        unit_scale=True,
        desc=dest.name,
    ) as bar:
        while True:
            chunk = response.read(1024 * 1024)
            if not chunk:
                break
            handle.write(chunk)
            bar.update(len(chunk))
    tmp.replace(dest)


if not stanford_ready():
    tar_path = STANFORD / "images.tar"
    download_file(IMAGES_URL, tar_path, min_bytes=600 * 1024 * 1024)
    print(f"распаковка {tar_path}")
    with tarfile.open(tar_path) as tar:
        try:
            tar.extractall(STANFORD, filter="data")
        except TypeError:
            tar.extractall(STANFORD)

if not stanford_ready():
    raise RuntimeError(f"В {IMAGES} нет полного Stanford Dogs.")

print(f"датасет: {IMAGES}")


In [ ]:
def list_images(folder: Path) -> list[Path]:
    if not folder.is_dir():
        return []
    return sorted(
        path
        for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png"}
    )


def to_rel(path: Path) -> str:
    return path.resolve().relative_to(BASE.resolve()).as_posix()


def from_rel(value: str) -> Path:
    return BASE / value


available = {path.name: path for path in IMAGES.iterdir() if path.is_dir()}
lower_map = {name.lower(): name for name in available}
resolved = []
for breed in KNOWN_BREEDS:
    if breed in available:
        resolved.append(breed)
    elif breed.lower() in lower_map:
        resolved.append(lower_map[breed.lower()])
    else:
        raise FileNotFoundError(f"Нет папки породы {breed} в {IMAGES}")

KNOWN_BREEDS = resolved
known_set = set(KNOWN_BREEDS)
UNSEEN_BREEDS = sorted(name for name in available if name not in known_set)
print(f"известных пород: {len(KNOWN_BREEDS)}, незнакомых: {len(UNSEEN_BREEDS)}")
if len(KNOWN_BREEDS) != 25:
    raise RuntimeError(f"Ожидалось 25 известных пород, получено {len(KNOWN_BREEDS)}.")


def split_breed(breed: str) -> dict[str, list[Path]]:
    paths = list_images(IMAGES / breed)
    rng = random.Random(f"{SEED}:{breed}")
    rng.shuffle(paths)
    count = len(paths)
    if count < 3:
        raise RuntimeError(f"У {breed} всего {count} изображений, для сплита мало.")
    n_train = max(1, int(round(count * TRAIN_RATIO)))
    n_val = max(1, int(round(count * VAL_RATIO)))
    while n_train + n_val >= count:
        if n_train >= n_val and n_train > 1:
            n_train -= 1
        elif n_val > 1:
            n_val -= 1
        else:
            raise RuntimeError(f"Не удалось разбить {breed} на train/val/test.")
    return {
        "train": paths[:n_train],
        "val": paths[n_train : n_train + n_val],
        "test": paths[n_train + n_val :],
    }


splits = {breed: split_breed(breed) for breed in KNOWN_BREEDS}
train_paths = {path.resolve() for breed in KNOWN_BREEDS for path in splits[breed]["train"]}
val_paths = {path.resolve() for breed in KNOWN_BREEDS for path in splits[breed]["val"]}
test_paths = {path.resolve() for breed in KNOWN_BREEDS for path in splits[breed]["test"]}
assert train_paths.isdisjoint(val_paths)
assert train_paths.isdisjoint(test_paths)
assert val_paths.isdisjoint(test_paths)

split_payload = {
    breed: {part: [to_rel(path) for path in paths] for part, paths in parts.items()}
    for breed, parts in splits.items()
}
(RESULTS / "splits.json").write_text(
    json.dumps(split_payload, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

split_counts = pd.DataFrame(
    [
        {
            "breed": short_name(breed),
            "train": len(parts["train"]),
            "val": len(parts["val"]),
            "test": len(parts["test"]),
        }
        for breed, parts in splits.items()
    ]
)
print(split_counts.to_string(index=False))
print(f"test кадров известных пород: {len(test_paths)}")


## Волк, лиса и не-собаки

Стартовый набор качается с Wikimedia Commons до обучения, чтобы отсутствие внешних кадров было видно сразу. Свои файлы можно просто положить в папки — они подхватятся вместе со стартовым набором:

- `dog_breeds/data/external/similar/wolf/`
- `dog_breeds/data/external/similar/fox/`
- `dog_breeds/data/external/not_dogs/<класс>/`

Списки кадров для инференса собираются здесь же. Незнакомые породы не входят в `splits`.


In [ ]:
STARTER_IMAGES = [
    ("similar/wolf", "Canis lupus lupus qtl1.jpg"),
    ("similar/wolf", "Front view of a resting Canis lupus ssp.jpg"),
    ("similar/wolf", "Adara (Canis lupus)2.jpg"),
    ("similar/fox", "Vulpes vulpes laying in snow.jpg"),
    ("similar/fox", "Vulpes vulpes sitting.jpg"),
    ("similar/fox", "Alaska Red Fox (Vulpes vulpes).jpg"),
    ("not_dogs/cat", "Cat November 2010-1a.jpg"),
    ("not_dogs/cat", "Felis catus-cat on snow.jpg"),
    ("not_dogs/cat", "Tired 20-year-old cat.jpg"),
    ("not_dogs/car", "Chevrolet-Cobalt-Coupe.JPG"),
    ("not_dogs/car", "1970 Nissan Skyline Sedan 2000GT front left.jpg"),
    ("not_dogs/car", "1957 Volkswagen Beetle, export model, in front of Porta Nigra in Trier 2023-05-01.jpg"),
    ("not_dogs/bird", "Erithacus rubecula with cocked head.jpg"),
    ("not_dogs/bird", "Tawny owl.jpg"),
    ("not_dogs/bird", "Cyanistes caeruleus 3 Luc Viatour.jpg"),
]


def local_image_name(filename: str) -> str:
    cleaned = []
    for char in Path(filename).name:
        cleaned.append(char if char.isalnum() or char in "._-" else "_")
    name = "".join(cleaned)
    while "__" in name:
        name = name.replace("__", "_")
    return name.strip("_")


def commons_file_url(filename: str) -> str:
    return (
        "https://commons.wikimedia.org/wiki/Special:FilePath/"
        + quote(filename, safe="")
        + "?width=800"
    )


starter_errors = []
for relative_dir, filename in STARTER_IMAGES:
    destination_dir = EXTERNAL / relative_dir
    destination_dir.mkdir(parents=True, exist_ok=True)
    destination = destination_dir / local_image_name(filename)
    if destination.exists() and destination.stat().st_size > 10_000:
        continue
    request = urllib.request.Request(
        commons_file_url(filename),
        headers={"User-Agent": USER_AGENT},
    )
    try:
        with urllib.request.urlopen(request, timeout=60) as response:
            destination.write_bytes(response.read())
        print(f"скачано: {destination.relative_to(BASE)}")
    except Exception as exc:
        starter_errors.append(f"{filename}: {exc}")
        if destination.exists() and destination.stat().st_size == 0:
            destination.unlink()

if starter_errors:
    print("Часть стартовых кадров не скачалась. Собачьи группы всё равно посчитаются.")
    for line in starter_errors:
        print(" ", line)


def image_record(path: Path, group: str, label: str) -> dict:
    return {"path": to_rel(path), "group": group, "label": label}


records = {key: [] for key in ("val_known", "known", "other_dogs", "similar", "not_dogs")}
for breed in KNOWN_BREEDS:
    label = short_name(breed)
    for path in splits[breed]["val"]:
        records["val_known"].append(image_record(path, "val_known", label))
    for path in splits[breed]["test"]:
        records["known"].append(image_record(path, "known", label))

other_rng = random.Random(f"{SEED}:other-dogs")
for breed in UNSEEN_BREEDS:
    files = list_images(IMAGES / breed)
    other_rng.shuffle(files)
    for path in files[:N_OTHER_PER_BREED]:
        records["other_dogs"].append(image_record(path, "other_dogs", short_name(breed)))

for kind in ("wolf", "fox"):
    for path in list_images(EXTERNAL / "similar" / kind):
        records["similar"].append(image_record(path, "similar", kind))

not_dogs_root = EXTERNAL / "not_dogs"
if not_dogs_root.is_dir():
    for class_dir in sorted(path for path in not_dogs_root.iterdir() if path.is_dir()):
        for path in list_images(class_dir):
            records["not_dogs"].append(image_record(path, "not_dogs", class_dir.name))

for key, rows in records.items():
    print(f"{key}: {len(rows)}")


## 25 моделей MobileNetV2

У каждой породы свой MobileNetV2 с весами ImageNet. Голова заменяется на один логит, лосс — бинарная кросс-энтропия с логитами. Обучаются голова и последний блок признаков, ранние слои заморожены. Аугментации: случайный кроп, отражение, цветовой джиттер.

Ранняя остановка — по ROC-AUC на сбалансированной val-выборке этой бинарной задачи (порода против такого же числа кадров остальных 24). Это не тот val, по которому потом выбирается порог решения: порог считается по обычным val-кадрам известных пород, уже после обучения.

Чекпоинт: `dog_breeds/models/binary/<порода>.pt`.


In [ ]:
TRAIN_TF = T.Compose(
    [
        T.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)
EVAL_TF = T.Compose(
    [
        T.Resize(IMG_SIZE + 32),
        T.CenterCrop(IMG_SIZE),
        T.ToTensor(),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)


class ImageListDataset(Dataset):
    def __init__(self, samples: list[tuple[Path, float]], transform):
        self.samples = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, index: int):
        path, label = self.samples[index]
        with Image.open(path) as image:
            tensor = self.transform(image.convert("RGB"))
        return tensor, torch.tensor(label, dtype=torch.float32)


def sample_negatives(pools: dict[str, list[Path]], positive: str, n_neg: int, seed_key: str) -> list[Path]:
    rng = random.Random(seed_key)
    others = [breed for breed in KNOWN_BREEDS if breed != positive]
    shuffled = {}
    for breed in others:
        items = list(pools[breed])
        rng.shuffle(items)
        shuffled[breed] = items
    picked = []
    cursor = 0
    while len(picked) < n_neg:
        progress = False
        for breed in others:
            if len(picked) >= n_neg:
                break
            items = shuffled[breed]
            if cursor < len(items):
                picked.append(items[cursor])
                progress = True
        if not progress:
            break
        cursor += 1
    if len(picked) < n_neg:
        print(f"  предупреждение: для {short_name(positive)} негативов {len(picked)} вместо {n_neg}")
    return picked


def make_samples(breed: str, part: str) -> list[tuple[Path, float]]:
    positive = list(splits[breed][part])
    n_neg = int(round(len(positive) * NEG_PER_POS))
    pools = {name: splits[name][part] for name in KNOWN_BREEDS}
    negative = sample_negatives(pools, breed, n_neg, f"{SEED}:{part}:{breed}")
    return [(path, 1.0) for path in positive] + [(path, 0.0) for path in negative]


for breed in KNOWN_BREEDS:
    samples = make_samples(breed, "train")
    used = {path.resolve() for path, _ in samples}
    assert used.isdisjoint(test_paths)
    assert used.isdisjoint(val_paths)
    positives = {path.resolve() for path, label in samples if label == 1.0}
    negatives = [path.resolve() for path, label in samples if label == 0.0]
    allowed = {
        path.resolve()
        for other in KNOWN_BREEDS
        if other != breed
        for path in splits[other]["train"]
    }
    assert positives == {path.resolve() for path in splits[breed]["train"]}
    assert set(negatives) <= allowed
    assert len(negatives) == len(set(negatives))
    assert positives.isdisjoint(negatives)
print("проверка утечки: тестовые кадры не входят в обучение ни одной из 25 моделей")


def checkpoint_path(breed: str) -> Path:
    return MODELS / f"{short_name(breed)}.pt"


def load_checkpoint(path: Path) -> dict:
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def build_model(pretrained: bool) -> nn.Module:
    weights = MobileNet_V2_Weights.DEFAULT if pretrained else None
    model = mobilenet_v2(weights=weights)
    for param in model.parameters():
        param.requires_grad = False
    # features[-2] — последний inverted residual, features[-1] — свёртка 1x1 в 1280 каналов.
    for block in (model.features[-2], model.features[-1]):
        for param in block.parameters():
            param.requires_grad = True
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2),
        nn.Linear(model.last_channel, 1),
    )
    return model


def set_train_mode(model: nn.Module) -> None:
    model.train()
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            if not any(param.requires_grad for param in module.parameters()):
                module.eval()


@torch.no_grad()
def evaluate_auc(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    scores = []
    labels = []
    for images, batch_labels in loader:
        logits = model(images.to(DEVICE)).squeeze(1)
        scores.append(torch.sigmoid(logits).detach().cpu().numpy())
        labels.append(batch_labels.numpy())
    y_true = np.concatenate(labels)
    y_score = np.concatenate(scores)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(roc_auc_score(y_true, y_score))


def train_one(breed: str) -> dict:
    path = checkpoint_path(breed)
    name = short_name(breed)
    if path.exists() and not FORCE_RETRAIN:
        payload = load_checkpoint(path)
        print(f"{name}: чекпоинт уже есть, val ROC-AUC={payload.get('val_auc')}")
        return {
            "breed": name,
            "val_auc": payload.get("val_auc"),
            "epoch": payload.get("epoch"),
            "skipped": True,
        }

    train_samples = make_samples(breed, "train")
    val_samples = make_samples(breed, "val")
    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(
        ImageListDataset(train_samples, TRAIN_TF),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    val_loader = DataLoader(
        ImageListDataset(val_samples, EVAL_TF),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )
    set_seed(SEED)
    model = build_model(pretrained=True).to(DEVICE)
    optimizer = torch.optim.Adam(
        [param for param in model.parameters() if param.requires_grad],
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.BCEWithLogitsLoss()
    n_pos = sum(label for _, label in train_samples)
    print(f"{name}: train {len(train_samples)} (pos {n_pos:.0f}), val {len(val_samples)}")

    best_auc = -1.0
    best_epoch = 0
    best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    wait = 0
    for epoch in range(1, EPOCHS + 1):
        set_train_mode(model)
        losses = []
        for images, labels in train_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        auc = evaluate_auc(model, val_loader)
        mean_loss = float(np.mean(losses)) if losses else float("nan")
        auc_text = "nan" if np.isnan(auc) else f"{auc:.4f}"
        print(f"  epoch {epoch}: loss={mean_loss:.4f} val_auc={auc_text}")
        if not np.isnan(auc) and auc > best_auc + 1e-4:
            best_auc = auc
            best_epoch = epoch
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= PATIENCE:
                print(f"  early stopping на эпохе {epoch}")
                break

    payload = {
        "breed": breed,
        "state_dict": best_state,
        "val_auc": None if best_auc < 0 else best_auc,
        "epoch": best_epoch,
    }
    torch.save(payload, path)
    return {"breed": name, "val_auc": payload["val_auc"], "epoch": best_epoch, "skipped": False}


def load_trained_model(breed: str) -> nn.Module:
    path = checkpoint_path(breed)
    if not path.exists():
        raise FileNotFoundError(f"Нет чекпоинта {path}. Сначала запустите обучение.")
    model = build_model(pretrained=False)
    model.load_state_dict(load_checkpoint(path)["state_dict"])
    model.to(DEVICE)
    model.eval()
    return model


probe = build_model(pretrained=False)
n_trainable = sum(param.numel() for param in probe.parameters() if param.requires_grad)
n_total = sum(param.numel() for param in probe.parameters())
print(f"обучаемых параметров в одной модели: {n_trainable} из {n_total}")
assert probe.classifier[-1].out_features == 1
del probe

auc_rows = [train_one(breed) for breed in KNOWN_BREEDS]
auc_df = pd.DataFrame(auc_rows)
auc_df.to_csv(RESULTS / "val_auc.csv", index=False)
auc_df


## Инференс

Каждый кадр прогоняется через все 25 моделей. В CSV сохраняются путь, группа, истинная метка, 25 вероятностей, максимум, порода аргмакса и число моделей с вероятностью не ниже 0.5.

Порог 0.5 в колонке `n_fired` — только удобная отсечка по умолчанию. Метрики ниже пересчитывают решение для 0.5, 0.7, 0.9 и для порога, выбранного на val.


In [ ]:
@torch.no_grad()
def predict_paths(model: nn.Module, paths: list[Path], batch_size: int = INFER_BATCH) -> np.ndarray:
    model.eval()
    scores = np.full(len(paths), np.nan, dtype=np.float64)
    batch_index: list[int] = []
    batch_tensors = []

    def flush() -> None:
        if not batch_tensors:
            return
        logits = model(torch.stack(batch_tensors).to(DEVICE)).squeeze(1)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        for index, prob in zip(batch_index, probs):
            scores[index] = float(prob)
        batch_index.clear()
        batch_tensors.clear()

    for index, path in enumerate(paths):
        try:
            with Image.open(path) as image:
                batch_tensors.append(EVAL_TF(image.convert("RGB")))
            batch_index.append(index)
        except Exception as exc:
            print(f"пропуск {path}: {exc}")
            continue
        if len(batch_tensors) == batch_size:
            flush()
    flush()
    return scores


def free_memory() -> None:
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps" and hasattr(torch, "mps"):
        torch.mps.empty_cache()


eval_order = ["val_known", "known", "other_dogs", "similar", "not_dogs"]
eval_rows = [row for key in eval_order for row in records[key]]
if not eval_rows:
    raise RuntimeError("Нет изображений для инференса.")

eval_paths = [from_rel(row["path"]) for row in eval_rows]
short_names = [short_name(breed) for breed in KNOWN_BREEDS]
prob_cols = [f"p_{name}" for name in short_names]
prob_matrix = np.full((len(eval_rows), len(KNOWN_BREEDS)), np.nan, dtype=np.float64)

for column, breed in enumerate(tqdm(KNOWN_BREEDS, desc="inference")):
    model = load_trained_model(breed)
    prob_matrix[:, column] = predict_paths(model, eval_paths)
    del model
    free_memory()

probs_df = pd.DataFrame(eval_rows)
for column, values in zip(prob_cols, prob_matrix.T):
    probs_df[column] = values

values = probs_df[prob_cols].to_numpy(dtype=float)
all_nan = np.isnan(values).all(axis=1)
filled = np.where(np.isnan(values), -np.inf, values)
max_index = np.zeros(len(probs_df), dtype=int)
max_p = np.full(len(probs_df), np.nan)
if (~all_nan).any():
    max_index[~all_nan] = filled[~all_nan].argmax(axis=1)
    max_p[~all_nan] = values[~all_nan, max_index[~all_nan]]

probs_df["max_p"] = max_p
probs_df["argmax_breed"] = [short_names[index] for index in max_index]
probs_df.loc[all_nan, "argmax_breed"] = pd.NA
probs_df["n_fired"] = (values >= 0.5).sum(axis=1).astype(int)

file_by_group = {
    "val_known": "probs_val_known.csv",
    "known": "probs_known.csv",
    "other_dogs": "probs_other_dogs.csv",
    "similar": "probs_similar.csv",
    "not_dogs": "probs_not_dogs.csv",
}
for group_name, filename in file_by_group.items():
    part = probs_df.loc[probs_df["group"] == group_name].copy()
    part.to_csv(RESULTS / filename, index=False, float_format="%.6f")
    print(f"{filename}: {len(part)}")


## Решение и метрики

Правило задано до просмотра незнакомых групп:

- если ни одна вероятность не выше порога, ответ — «другие»;
- иначе ответ — порода с максимальной вероятностью;
- `n_fired` — сколько моделей одновременно перешли порог.

Фиксированные пороги: 0.5, 0.7 и 0.9. На незнакомых группах порог не подбирается.

Дополнительный порог ищется только по val известных пород: это наибольший порог, при котором доля кадров с верной породой не ниже 0.8. Кадр, ушедший в «другие», для recall считается промахом. Тот же порог затем применяется ко всем тестовым группам отдельной строкой отчёта.

Для известных пород `argmax_acc` — доля кадров, где порода с максимальной вероятностью совпала с истиной, без порога. `other_rate` на известных породах — как часто свой кадр отвергнут как «другие». Для остальных групп `other_rate` — основная доля успеха.


In [ ]:
def load_probs(filename: str) -> pd.DataFrame:
    path = RESULTS / filename
    if not path.exists():
        raise FileNotFoundError(f"Нет {path}. Сначала запустите инференс.")
    return pd.read_csv(path)


val_df = load_probs("probs_val_known.csv")
groups = {
    "known": load_probs("probs_known.csv"),
    "other_dogs": load_probs("probs_other_dogs.csv"),
    "similar": load_probs("probs_similar.csv"),
    "not_dogs": load_probs("probs_not_dogs.csv"),
}
prob_cols = [column for column in val_df.columns if column.startswith("p_")]


def decide(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    out = df.copy()
    probs = out[prob_cols].to_numpy(dtype=float)
    max_p = out["max_p"].to_numpy(dtype=float)
    pred = np.where(max_p >= threshold, out["argmax_breed"].astype(str), OTHER)
    pred = np.where(np.isnan(max_p), OTHER, pred)
    out["pred"] = pred
    out["n_fired_thr"] = (probs >= threshold).sum(axis=1)
    return out


def known_recall(df: pd.DataFrame, threshold: float) -> float:
    if df.empty:
        return float("nan")
    decided = decide(df, threshold)
    return float((decided["pred"].to_numpy() == decided["label"].to_numpy()).mean())


grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
recall_grid = [(float(threshold), known_recall(val_df, float(threshold))) for threshold in grid]
eligible = [threshold for threshold, recall in recall_grid if recall >= TARGET_RECALL]
if eligible:
    val_threshold = max(eligible)
else:
    val_threshold = max(recall_grid, key=lambda item: (item[1], -item[0]))[0]
    print(
        f"На val ни один порог не даёт recall >= {TARGET_RECALL:.2f}. "
        "Берётся порог с наибольшим recall."
    )
val_recall = known_recall(val_df, val_threshold)
print(
    f"порог по val: {val_threshold:.2f}, "
    f"recall известных пород на val: {val_recall:.3f} (цель >= {TARGET_RECALL:.2f})"
)


def empty_summary(group_name: str, threshold: float, rule: str) -> dict:
    return {
        "group": group_name,
        "rule": rule,
        "threshold": threshold,
        "n": 0,
        "other_rate": None,
        "mean_max_p": None,
        "median_max_p": None,
        "multi_fired_rate": None,
        "argmax_acc": None,
        "top_mean_breed": "",
        "top_mean_p": None,
        "top_false_breed": "",
        "top_false_count": 0,
    }


def summarize(df: pd.DataFrame, threshold: float, group_name: str, rule: str) -> dict:
    if df.empty:
        return empty_summary(group_name, threshold, rule)
    decided = decide(df, threshold)
    means = decided[prob_cols].mean(numeric_only=True)
    top_col = str(means.idxmax())
    if group_name == "known":
        false_mask = (decided["pred"] != OTHER) & (decided["pred"] != decided["label"])
        argmax_acc = float((decided["argmax_breed"] == decided["label"]).mean())
    else:
        false_mask = decided["pred"] != OTHER
        argmax_acc = None
    false_preds = decided.loc[false_mask, "pred"]
    if false_preds.empty:
        top_false = ""
        top_false_count = 0
    else:
        counts = false_preds.value_counts()
        top_false = str(counts.index[0])
        top_false_count = int(counts.iloc[0])
    return {
        "group": group_name,
        "rule": rule,
        "threshold": float(threshold),
        "n": int(len(decided)),
        "other_rate": float((decided["pred"] == OTHER).mean()),
        "mean_max_p": float(decided["max_p"].mean()),
        "median_max_p": float(decided["max_p"].median()),
        "multi_fired_rate": float((decided["n_fired_thr"] > 1).mean()),
        "argmax_acc": argmax_acc,
        "top_mean_breed": top_col[2:],
        "top_mean_p": float(means.max()),
        "top_false_breed": top_false,
        "top_false_count": top_false_count,
    }


detail = {
    "known": groups["known"],
    "other_dogs": groups["other_dogs"],
}
if groups["similar"].empty:
    detail["similar"] = groups["similar"]
else:
    for label, part in groups["similar"].groupby("label"):
        detail[str(label)] = part
if groups["not_dogs"].empty:
    detail["not_dogs"] = groups["not_dogs"]
else:
    for label, part in groups["not_dogs"].groupby("label"):
        detail[f"not_dogs/{label}"] = part

rules = [(float(threshold), "fixed") for threshold in THRESHOLDS]
rules.append((float(val_threshold), "val_recall"))
metric_rows = [
    summarize(df, threshold, group_name, rule)
    for threshold, rule in rules
    for group_name, df in detail.items()
]
metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(RESULTS / "metrics_by_threshold.csv", index=False)
metrics_df


In [ ]:
plt.rcParams["axes.unicode_minus"] = False

hist_source = [
    ("известные породы", groups["known"]),
    ("другие породы", groups["other_dogs"]),
    ("волк и лиса", groups["similar"]),
    ("не собаки", groups["not_dogs"]),
]
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)
for ax, (title, df) in zip(axes.ravel(), hist_source):
    ax.set_title(title)
    ax.set_xlim(0, 1)
    if df.empty or df["max_p"].dropna().empty:
        ax.text(0.5, 0.5, "нет кадров", ha="center", va="center")
        continue
    ax.hist(df["max_p"].dropna(), bins=np.linspace(0, 1, 21), color="#3c6e71")
    ax.set_xlabel("максимальная вероятность")
    ax.set_ylabel("кадры")
fig.tight_layout()
fig.savefig(RESULTS / "max_p_hist.png", dpi=140)
plt.show()


def heatmap(row_labels: list[str], matrix: np.ndarray, filename: str, title: str) -> None:
    if not row_labels:
        print(f"пропуск графика {filename}: нет строк")
        return
    fig_w = max(12, 0.45 * matrix.shape[1])
    fig_h = max(3.2, 0.45 * matrix.shape[0] + 1.5)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    image = ax.imshow(matrix, aspect="auto", vmin=0, vmax=1, cmap="magma")
    ax.set_title(title)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels)
    ax.set_xticks(range(len(prob_cols)))
    ax.set_xticklabels([column[2:] for column in prob_cols], rotation=90, fontsize=8)
    fig.colorbar(image, ax=ax, fraction=0.02, pad=0.02, label="средняя вероятность")
    fig.tight_layout()
    fig.savefig(RESULTS / filename, dpi=140)
    plt.show()


heat_labels = []
heat_rows = []
for title, df in hist_source:
    if df.empty:
        continue
    heat_labels.append(title)
    heat_rows.append(df[prob_cols].mean().to_numpy(dtype=float))
heatmap(
    heat_labels,
    np.vstack(heat_rows) if heat_rows else np.empty((0, len(prob_cols))),
    "mean_prob_heatmap.png",
    "Средние вероятности 25 моделей по группам",
)

known_df = groups["known"]
if not known_df.empty:
    breed_labels = []
    breed_rows = []
    for label in [short_name(breed) for breed in KNOWN_BREEDS]:
        part = known_df.loc[known_df["label"] == label]
        if part.empty:
            continue
        breed_labels.append(label)
        breed_rows.append(part[prob_cols].mean().to_numpy(dtype=float))
    heatmap(
        breed_labels,
        np.vstack(breed_rows),
        "known_breed_heatmap.png",
        "Известные породы: средняя вероятность каждой модели",
    )

present = [(title, df) for title, df in hist_source if not df.empty and df["max_p"].notna().any()]
if present:
    fig, axes = plt.subplots(len(present), 1, figsize=(14, 3.4 * len(present)), squeeze=False)
    for ax, (title, df) in zip(axes[:, 0], present):
        row = df.loc[df["max_p"].idxmax()]
        ax.bar([column[2:] for column in prob_cols], row[prob_cols].to_numpy(dtype=float), color="#3c6e71")
        ax.axhline(0.5, color="gray", ls="--", lw=1)
        ax.set_ylim(0, 1)
        ax.tick_params(axis="x", labelrotation=90, labelsize=8)
        ax.set_ylabel("P(порода)")
        ax.set_title(
            f"{title}: истина={row['label']}, argmax={row['argmax_breed']}, max={row['max_p']:.2f}"
        )
    fig.tight_layout()
    fig.savefig(RESULTS / "examples_bars.png", dpi=140)
    plt.show()
else:
    print("нет кадров для примеров вероятностей")


## Черновик вывода

Текст ниже собирается из посчитанных метрик. Его можно править перед письмом: цифры берутся из `dog_breeds/results/metrics_by_threshold.csv`, вероятности — из `probs_*.csv`.

Смотреть в первую очередь, низкая ли уверенность у всех 25 моделей на неизвестном кадре или какая-то одна всё равно уверенно узнаёт свою породу. Для этого в каждой группе есть средняя максимальная вероятность, порода с самой высокой средней вероятностью и порода, которая чаще всего получает кадр, не отвергнутый как «другие».


In [ ]:
def json_ready(value):
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    if isinstance(value, (np.floating, np.integer)):
        return value.item()
    return value


def interpretation(other_rate: float, top_mean_p: float) -> str:
    if other_rate >= 0.8 and top_mean_p < 0.4:
        return "в среднем все 25 моделей держат низкую уверенность, и класс «другие» отделяется."
    if top_mean_p >= 0.7 or other_rate < 0.5:
        return "какая-то модель всё равно уверенно относит кадры к своей породе."
    return "картина смешанная: часть кадров отвергается, но лидирующая модель не выглядит неуверенной."


def fmt_pct(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "нет кадров"
    return f"{100 * float(value):.1f}%"


def fmt_num(value) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "—"
    return f"{float(value):.3f}"


def row_for(group_name: str, rule: str, threshold=None):
    part = metrics_df.loc[(metrics_df["group"] == group_name) & (metrics_df["rule"] == rule)]
    if threshold is not None:
        part = part.loc[np.isclose(part["threshold"].to_numpy(dtype=float), threshold)]
    if part.empty:
        return None
    return part.iloc[0]


def describe(group_name: str, title: str, rule: str, threshold: float) -> str:
    row = row_for(group_name, rule, threshold)
    if row is None or int(row["n"]) == 0:
        return f"{title}: кадров нет."
    other_rate = float(row["other_rate"])
    top_mean_p = float(row["top_mean_p"])
    lines = [
        (
            f"{title}, n={int(row['n'])}: доля «другие» {fmt_pct(other_rate)}, "
            f"средняя max-вероятность {fmt_num(row['mean_max_p'])}, "
            f"медиана {fmt_num(row['median_max_p'])}."
        ),
        (
            f"Самая высокая средняя вероятность у {row['top_mean_breed']} "
            f"({fmt_num(top_mean_p)})."
        ),
    ]
    lines.append(
        f"Доля кадров, где порог перешли сразу несколько моделей: {fmt_pct(row['multi_fired_rate'])}."
    )
    if group_name == "known":
        if row["top_false_breed"]:
            lines.append(
                f"Чаще всего ошибочно называется {row['top_false_breed']} "
                f"({int(row['top_false_count'])} из {int(row['n'])})."
            )
        else:
            lines.append("Чужой породой не назван ни один кадр.")
        lines.append(
            f"Argmax совпал с истинной породой в {fmt_pct(row['argmax_acc'])} кадров, "
            f"как «другие» отвергнуто {fmt_pct(row['other_rate'])}."
        )
    else:
        if row["top_false_breed"]:
            lines.append(
                f"Среди кадров, которые не ушли в «другие», чаще всего названа порода "
                f"{row['top_false_breed']} ({int(row['top_false_count'])} из {int(row['n'])})."
            )
        else:
            lines.append("Ни один кадр не принят как известная порода.")
        lines.append(interpretation(other_rate, top_mean_p))
    return " ".join(lines)


auc_path = RESULTS / "val_auc.csv"
if auc_path.exists():
    auc_values = pd.read_csv(auc_path)["val_auc"].dropna()
    auc_sentence = (
        f"Медианный ROC-AUC бинарных моделей на их сбалансированном val — {float(auc_values.median()):.3f}."
        if not auc_values.empty
        else "ROC-AUC бинарных моделей в val_auc.csv не заполнен."
    )
else:
    auc_sentence = "Файл val_auc.csv не найден."

GROUP_TITLES = {
    "known": "Известные породы",
    "other_dogs": "Другие породы собак",
    "similar": "Похожие животные",
    "wolf": "Волк",
    "fox": "Лиса",
    "not_dogs": "Не собаки",
}


def group_title(group_name: str) -> str:
    if group_name in GROUP_TITLES:
        return GROUP_TITLES[group_name]
    if group_name.startswith("not_dogs/"):
        return "Не собаки: " + group_name.split("/", 1)[1]
    return group_name


paragraphs = [
    (
        "Проверил 25 отдельных бинарных MobileNetV2: каждая модель отличает свою породу "
        "от остальных 24 известных. Негативы брались только из этих 24, поэтому 95 других пород "
        "Stanford Dogs, волк, лиса и не-собаки в обучении не участвовали. "
        "Сохранены вероятности всех 25 классификаторов."
    ),
    auc_sentence,
    (
        f"Порог по val известных пород: {val_threshold:.2f} "
        f"(recall на val {val_recall:.3f}, цель не ниже {TARGET_RECALL:.2f}). "
        "На незнакомых группах порог не подбирался."
    ),
    "Порог 0.5.",
]
paragraphs.extend(
    describe(group_name, group_title(group_name), "fixed", 0.5) for group_name in detail
)
paragraphs.append(f"Порог по val, {val_threshold:.2f}.")
paragraphs.extend(
    describe(group_name, group_title(group_name), "val_recall", float(val_threshold))
    for group_name in detail
)
draft = "\n\n".join(paragraphs)
print(draft)
(RESULTS / "draft_conclusion.txt").write_text(draft + "\n", encoding="utf-8")

summary = {
    "device": str(DEVICE),
    "seed": SEED,
    "epochs": EPOCHS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "neg_per_pos": NEG_PER_POS,
    "img_size": IMG_SIZE,
    "thresholds": [float(threshold) for threshold in THRESHOLDS],
    "val_threshold": float(val_threshold),
    "val_recall": float(val_recall),
    "target_recall": TARGET_RECALL,
    "n_other_per_breed": N_OTHER_PER_BREED,
    "known_breeds": [short_name(breed) for breed in KNOWN_BREEDS],
    "unseen_breed_count": len(UNSEEN_BREEDS),
    "metrics": [{key: json_ready(value) for key, value in row.items()} for row in metric_rows],
}
(RESULTS / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"черновик: {RESULTS / 'draft_conclusion.txt'}")
